# DACN — checkpoint Gaussian batch8 epoch20 → private validation pairs → CPU diagnostic
This notebook performs inference only, never training or held-out-test evaluation.
Select a GPU runtime you can access. Availability/cost depends on your Colab account.
Use your own HF read access; enter token through getpass, never paste it in a cell.
No patient images or sample UUIDs are displayed. NPZ remains private in this runtime;
only download/share it privately with authorized project members. Runtime can reset,
so save artifacts before disconnecting. Metrics are exploratory validation candidates.


In [ ]:
import subprocess
import sys
from pathlib import Path

SOURCE_REVISION = "7a0f024d8a0027dfdd90d3eb3d45b4381105a85f"
REPO = Path("/content/DACN")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/tanh1c/DACN.git", str(REPO)], check=True)
if subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip():
    raise RuntimeError("Existing checkout has local changes; use a fresh Colab runtime")
subprocess.run(["git", "fetch", "origin", SOURCE_REVISION], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", SOURCE_REVISION], cwd=REPO, check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[train,hub,data]"], check=True
)
sys.path.insert(0, str(REPO / "src"))

In [ ]:
import hashlib
import json
from getpass import getpass

from huggingface_hub import HfApi, hf_hub_download, snapshot_download

hf_token = getpass("HF token with read access to private DACN repos: ")
api = HfApi(token=hf_token)
api.whoami()
DATA_REV = "350c74f2313e4e73ca65e7ffc6e9dcf74040b3b7"
MODEL_REV = "1788b1232aa19b3c9ae96aa95a74e2975f4b592a"
META_SHA = "9c60fac0ff2a2d6274e82db466e5e338303d0dc24986b3bab4b32b6e710af218"
CKPT_SHA = "b4fc177db493fce2143a6412161068c7af73a25e8ae7ef9d88c117863c7e37f1"
DATA_REPO = "dacn-bmad-team/bmad-flair-patient-disjoint-v1"
MODEL_REPO = "dacn-bmad-team/dacn-runs"
checkpoint_path = hf_hub_download(
    MODEL_REPO,
    "runs/ddpm-partial-t4-b8-e30-cont-20261102T034644Z/ddpm_epoch_020.pt",
    revision=MODEL_REV,
    token=hf_token,
)
checkpoint_bytes = Path(checkpoint_path).read_bytes()
if hashlib.sha256(checkpoint_bytes).hexdigest() != CKPT_SHA:
    raise ValueError("Checkpoint checksum mismatch")
DATA = Path("/content/private_dacn_validation")
snapshot_download(
    DATA_REPO,
    repo_type="dataset",
    revision=DATA_REV,
    local_dir=str(DATA),
    token=hf_token,
    allow_patterns=["dataset_metadata.json", "data/val/*"],
)
from dacn.evaluation.artifacts import _validation_data

_validation_data(DATA, META_SHA)
del hf_token
print("Pinned checkpoint and validation checksums verified; no test downloaded.")

## Step 1: bounded smoke only
Set RUN_INFERENCE=True to check four validation reconstructions at t*=3.
This stops after smoke; it does not run the full cohort. No training or test access.
A start manifest is written before checkpoint loading. Manual interruption is recorded;
a runtime reset cannot execute cleanup and may leave the last status as started.
All artifacts remain local and private until explicitly saved elsewhere.

In [ ]:
RUN_INFERENCE = False
if not RUN_INFERENCE:
    raise RuntimeError("Set RUN_INFERENCE=True to explicitly run the four-sample smoke")

import io
import time
import uuid
from datetime import datetime, timezone
from importlib.metadata import version

import numpy as np
import torch

from dacn.evaluation.ddpm import _ddpm_backend, produce_validation_artifact

OUT = Path("/content/private_dacn_runs") / ("gaussian-e20-t300-" + uuid.uuid4().hex[:12])
OUT.mkdir(parents=True, exist_ok=False)
artifact = OUT / "validation_pairs.npz"
provenance = {
    "dataset_repo_id": DATA_REPO,
    "dataset_revision": DATA_REV,
    "dataset_metadata_sha256": META_SHA,
    "model_repo_id": MODEL_REPO,
    "model_revision": MODEL_REV,
    "checkpoint_sha256": CKPT_SHA,
    "code_revision": SOURCE_REVISION,
    "train_seed": 17,
    "sampling_seed": 17,
    "starting_timestep": 300,
}
(OUT / "provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
manifest = {
    "status": "started",
    "stage": "smoke",
    "scope": "validation inference only; test sealed",
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "provenance": provenance,
    "smoke_timestep": 3,
    "smoke_samples": 0,
}


def save_manifest():
    path = OUT / "inference_manifest.json"
    temporary = path.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(manifest, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


save_manifest()
started = time.perf_counter()
try:
    if not torch.cuda.is_available():
        raise RuntimeError("Select a GPU runtime before inference")
    if hashlib.sha256(checkpoint_bytes).hexdigest() != CKPT_SHA:
        raise ValueError("Checkpoint checksum mismatch")
    state = torch.load(io.BytesIO(checkpoint_bytes), map_location="cpu", weights_only=False)
    if (state["epoch"], state["optimizer_steps"], state["config"]["seed"]) != (20, 18760, 17):
        raise ValueError("Checkpoint epoch, optimizer budget or training seed mismatch")
    manifest["gpu"] = torch.cuda.get_device_name(0)
    manifest["packages"] = {
        name: version(name) for name in ["torch", "numpy", "scipy", "scikit-learn"]
    }
    save_manifest()
    adapter = _ddpm_backend(checkpoint_bytes, state["config"], "cuda")
    by_id, images, _, _ = _validation_data(DATA, META_SHA)
    if len(by_id) != 83:
        raise ValueError("Expected the complete 83-slice validation cohort")
    for sample_id, position in list(by_id.items())[:4]:
        output = adapter(
            images[position][None, None].copy(),
            sample_id=sample_id,
            starting_timestep=3,
            sampling_base_seed=17,
        )
        if (
            output.shape != (1, 1, 256, 256)
            or output.dtype != np.float32
            or not np.isfinite(output).all()
            or np.any(np.abs(output) > 1)
        ):
            raise ValueError("Invalid smoke reconstruction")
        manifest["smoke_samples"] += 1
    manifest.update(status="smoke_completed")
except BaseException as exc:
    manifest.update(
        status="interrupted" if isinstance(exc, KeyboardInterrupt) else "failed",
        error_class=type(exc).__name__,
    )
    raise
finally:
    manifest["smoke_elapsed_seconds"] = time.perf_counter() - started
    manifest["updated_at_utc"] = datetime.now(timezone.utc).isoformat()
    save_manifest()
print({"status": manifest["status"], "smoke_samples": manifest["smoke_samples"]})

## Step 2: full validation, separate opt-in
Review smoke first, then set RUN_FULL_VALIDATION=True for all83 at t*=300/seed17.
The 45-minute limit is checked between reconstruction calls, not a hard GPU shutdown.
Stop/disconnect the runtime yourself if needed. No automatic retry or Modal charges.


In [ ]:
RUN_FULL_VALIDATION = False
if not RUN_FULL_VALIDATION:
    raise RuntimeError("Set RUN_FULL_VALIDATION=True only after reviewing the smoke result")
if "manifest" not in globals() or manifest.get("status") != "smoke_completed":
    raise RuntimeError("A successful smoke in this runtime is required before full validation")
if artifact.exists():
    raise FileExistsError("Existing artifact: start a fresh smoke/run instead of overwriting")

manifest.update(status="started", stage="full_validation")
manifest["full_started_at_utc"] = datetime.now(timezone.utc).isoformat()
save_manifest()
full_started = time.perf_counter()
deadline = full_started + 45 * 60


def bounded_adapter(*args, **kwargs):
    # Cooperative guard: it cannot interrupt a single in-flight CUDA operation.
    if time.perf_counter() >= deadline:
        raise TimeoutError("Validation exceeded the 45-minute cooperative budget")
    result = adapter(*args, **kwargs)
    if time.perf_counter() >= deadline:
        raise TimeoutError("Validation exceeded the 45-minute cooperative budget")
    return result


try:
    receipt = produce_validation_artifact(DATA, artifact, provenance, bounded_adapter)
    manifest.update(status="completed", receipt=receipt)
except BaseException as exc:
    manifest.update(
        status="interrupted" if isinstance(exc, KeyboardInterrupt) else "failed",
        error_class=type(exc).__name__,
    )
    raise
finally:
    manifest["full_elapsed_seconds"] = time.perf_counter() - full_started
    manifest["updated_at_utc"] = datetime.now(timezone.utc).isoformat()
    save_manifest()
print({"status": manifest["status"], "samples": receipt["samples"]})

In [ ]:
if "manifest" not in globals() or manifest.get("status") != "completed":
    raise RuntimeError("Complete full validation before scoring")

from dacn.evaluation.diagnostics import compare

assert (
    hashlib.sha256(artifact.read_bytes()).hexdigest() == receipt["reconstruction_artifact_sha256"]
)
report = compare(artifact, DATA, provenance)
(OUT / "diagnostic_report.json").write_text(json.dumps(report, indent=2, allow_nan=False))
for method, variants in report["results"].items():
    print(
        method,
        {
            variant: {
                "pixel_AP": variants[variant]["metrics"]["pixel"]["average_precision"],
                "pixel_Dice": variants[variant]["metrics"]["pixel"]["dice_micro"],
                "image_AUROC": variants[variant]["metrics"]["image"]["auroc"],
            }
            for variant in ["baseline", "foreground_floor"]
        },
    )
print("Report saved. Thresholds are not frozen; do not open test.")

## Private handoff
Download the aggregate diagnostic report, provenance and inference manifest.
The separate NPZ contains medical-derived arrays and private UUIDs; download it
only to a private project folder, never attach it to GitHub or Linear. Share
through an authorized private channel. Add actual run status/source/environment
and aggregate evidence to RESULT_TRACKING; do not report this notebook as run
until the inference cell has really completed.


In [ ]:
from google.colab import files

files.download(str(OUT / "diagnostic_report.json"))
files.download(str(OUT / "provenance.json"))
files.download(str(OUT / "inference_manifest.json"))
# Explicit private download if needed for B's local CPU analysis:
# files.download(str(artifact))